In [ ]:
# Llama-3.2 is gated. Store a Kaggle secret named HF_TOKEN.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login: OK')
except Exception as exc:
    print('HF login was not completed automatically:', exc)
    print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
# TODO for Question 4
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""
def analyze_query_weights(model,layer):
    q_weight = model.model.layers[layer].self_attn.q_proj.weight.detach()
    U,S,Vh = torch.linalg.svd(q_weight.float(),full_matrices=False)
    energy_ratios = torch.cumsum(S**2,dim=0)/torch.sum(S**2) * 100
    min_rank = int((energy_ratios>=50).nonzero()[0]+1)
    return S, energy_ratios, min_rank
    
for layer in [0,7,15]:
    S, energy, min_rank = analyze_query_weights(model,layer)
    print(f"Layer {layer}: min rank={min_rank}")
    plt.figure()
    plt.bar(range(1,201),S[:200].numpy(),width=1.0)
    plt.title(f"Layer {layer}: Singular Value Magnitude")
    plt.xlabel("Singular Value Index")
    plt.ylabel("Singular Value Magnitude")
    plt.grid(True)
    plt.gca().set_axisbelow(True)
    plt.savefig(f"/kaggle/working/q4_layer_{layer}_singular_values.png")
    plt.show()
    

    plt.figure()
    plt.plot(range(1,201),energy[:200].numpy())
    plt.title(f"Layer {layer}: Cumulative Energy")
    plt.xlabel("Target Rank")
    plt.ylabel("Cumulative Energy (%)")
    plt.grid(True)
    plt.gca().set_axisbelow(True)
    plt.savefig(f"/kaggle/working/q4_layer_{layer}_cumulative_energy.png")
    plt.show()
    

